# 03 - Model Registry

**Goal:** Learn to version, stage, and manage trained models so you always know which model is in production.

---

## The Problem

Your current model storage:

```
weights/
├── training_spine_semantic_new_v8/
│   └── best_metric_model.pth        ← which experiment produced this?
└── training_spine_instance_new_v2/
    └── best_metric_model.pth        ← is this the one in Docker?
```

Questions nobody can answer:
- Which `best_metric_model.pth` is deployed in production?
- What was the Dice score when this model was validated?
- Can we roll back to the previous model if this one has a bug?

A **Model Registry** is a catalog that tracks every model version with its metadata.

## Model Lifecycle

```
Training Run → Register → Stage → Deploy → (Archive)

┌─────────┐    ┌──────────┐    ┌───────────┐    ┌────────────┐
│ MLflow   │───→│ Registry │───→│  Staging  │───→│ Production │
│ Run      │    │ v1, v2.. │    │ (testing) │    │ (deployed) │
└─────────┘    └──────────┘    └───────────┘    └────────────┘
                                                       │
                                                       ▼
                                                ┌────────────┐
                                                │  Archived  │
                                                │ (old vers) │
                                                └────────────┘
```

| Stage | Meaning |
|-------|---------|
| **None** | Just registered, not evaluated yet |
| **Staging** | Being tested / validated |
| **Production** | Currently deployed (in Docker image) |
| **Archived** | Retired, kept for reference |

In [ ]:
import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import os
import shutil

# Fresh MLflow tracking directory
TRACKING_DIR = '/tmp/mlflow-registry-demo'
if os.path.exists(TRACKING_DIR):
    shutil.rmtree(TRACKING_DIR)

mlflow.set_tracking_uri(f"file://{TRACKING_DIR}")
mlflow.set_experiment("spine-segmentation")

print(f"MLflow tracking: {TRACKING_DIR}")

In [ ]:
# Same simple model from notebook 02
class MiniUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=4, features=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, features, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(features, features * 2, 3, padding=1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(features * 2, features, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(features, out_channels, 1),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

## Train Two Model Versions

Let's simulate two training runs with different results.

In [ ]:
def train_and_log(run_name, lr, features, simulated_dice):
    """Train a model and log to MLflow (with simulated metrics)."""
    
    with mlflow.start_run(run_name=run_name) as run:
        # Log params
        mlflow.log_param("lr", lr)
        mlflow.log_param("features", features)
        
        # Create and "train" model
        model = MiniUNet(features=features)
        # (in reality: actual training loop here)
        
        # Log simulated metrics
        mlflow.log_metric("val_dice_mean", simulated_dice)
        mlflow.log_metric("val_dice_cervical", simulated_dice + 0.02)
        mlflow.log_metric("val_dice_thoracic", simulated_dice - 0.01)
        mlflow.log_metric("val_dice_lumbar", simulated_dice + 0.01)
        
        # Log model using MLflow's PyTorch integration
        mlflow.pytorch.log_model(model, "model")
        
        print(f"Run '{run_name}' | dice={simulated_dice:.3f} | run_id={run.info.run_id[:8]}")
        return run.info.run_id


# Version 1: baseline model
run_v1 = train_and_log("v8-baseline", lr=0.001, features=24, simulated_dice=0.847)

# Version 2: improved model (more features, tuned LR)
run_v2 = train_and_log("v9-improved", lr=0.0005, features=32, simulated_dice=0.891)

## Register Models

Now we promote these runs to the Model Registry.

In [ ]:
# Register model from run v1
model_name = "spine-semantic-model"

result_v1 = mlflow.register_model(
    model_uri=f"runs:/{run_v1}/model",
    name=model_name,
)

print(f"Registered: {result_v1.name} version {result_v1.version}")

In [ ]:
# Register model from run v2 (automatically becomes version 2)
result_v2 = mlflow.register_model(
    model_uri=f"runs:/{run_v2}/model",
    name=model_name,
)

print(f"Registered: {result_v2.name} version {result_v2.version}")

In [ ]:
# List all versions of this model
client = mlflow.tracking.MlflowClient()

for mv in client.search_model_versions(f"name='{model_name}'"):
    # Get the source run to show metrics
    run = client.get_run(mv.run_id)
    dice = run.data.metrics.get("val_dice_mean", "N/A")
    print(f"  Version {mv.version} | dice={dice} | stage={mv.current_stage} | run={mv.run_id[:8]}")

## Manage Model Stages

Move models through the lifecycle: None → Staging → Production.

In [ ]:
# Move v1 to Production (it was our first deployed model)
client.transition_model_version_stage(
    name=model_name,
    version="1",
    stage="Production",
)
print("v1 → Production")

# Move v2 to Staging (testing it first)
client.transition_model_version_stage(
    name=model_name,
    version="2",
    stage="Staging",
)
print("v2 → Staging")

In [ ]:
# View current state
for mv in client.search_model_versions(f"name='{model_name}'"):
    run = client.get_run(mv.run_id)
    dice = run.data.metrics.get("val_dice_mean", "N/A")
    print(f"  Version {mv.version} | dice={dice} | stage={mv.current_stage}")

In [ ]:
# v2 passed testing! Promote to Production, archive v1
client.transition_model_version_stage(
    name=model_name,
    version="2",
    stage="Production",
)

client.transition_model_version_stage(
    name=model_name,
    version="1",
    stage="Archived",
)

print("\nUpdated stages:")
for mv in client.search_model_versions(f"name='{model_name}'"):
    run = client.get_run(mv.run_id)
    dice = run.data.metrics.get("val_dice_mean", "N/A")
    print(f"  Version {mv.version} | dice={dice} | stage={mv.current_stage}")

## Load Model by Name (Not by File Path)

This is the key benefit. Instead of:
```python
# Current (fragile)
model.load_state_dict(torch.load("weights/training_spine_semantic_new_v8/best_metric_model.pth"))
```

You do:
```python
# With registry (robust)
model = mlflow.pytorch.load_model("models:/spine-semantic-model/Production")
```

In [ ]:
# Load the production model by name
production_model = mlflow.pytorch.load_model(
    f"models:/{model_name}/Production"
)

print(f"Loaded production model: {type(production_model).__name__}")
print(f"Parameters: {sum(p.numel() for p in production_model.parameters()):,}")

# Test it
dummy_input = torch.randn(1, 1, 64, 64)
output = production_model(dummy_input)
print(f"Output shape: {output.shape}")

In [ ]:
# Load by specific version (for rollback)
v1_model = mlflow.pytorch.load_model(
    f"models:/{model_name}/1"
)
print(f"Rolled back to v1: {sum(p.numel() for p in v1_model.parameters()):,} params")

## Add Descriptions (Documentation)

You can annotate models with descriptions — useful for audits.

In [ ]:
# Add description to the registered model
client.update_registered_model(
    name=model_name,
    description="Spine semantic segmentation (region model). "
                "Identifies cervical, thoracic, lumbar, sacrum regions."
)

# Add description to specific version
client.update_model_version(
    name=model_name,
    version="2",
    description="Improved model with 32 features. "
                "Trained on dataset v1 (20 patients), split 101225_s1ok. "
                "Mean Dice: 0.891."
)

# Read it back
mv = client.get_model_version(model_name, "2")
print(f"Model: {mv.name}")
print(f"Version: {mv.version}")
print(f"Stage: {mv.current_stage}")
print(f"Description: {mv.description}")

## Connection to Your Production Code

Currently in `main_pipeline.py`, models are loaded by file path:

```python
# Current: hardcoded paths
model_weights_region = "weights/training_spine_semantic_new_v8/best_metric_model.pth"
model_weights_instance = "weights/training_spine_instance_new_v2/best_metric_model.pth"
```

With a model registry, the inference pipeline becomes:

```python
# With registry: load by name and stage
region_model = mlflow.pytorch.load_model("models:/spine-semantic-model/Production")
instance_model = mlflow.pytorch.load_model("models:/spine-instance-model/Production")
```

**Benefits for FDA 510(k):**

| Question | Without Registry | With Registry |
|----------|-----------------|---------------|
| Which model is deployed? | Check Docker image | Query: `models:/name/Production` |
| What was its validation Dice? | Unknown | Linked to MLflow run |
| Who approved it for production? | Nobody tracked this | Stage transition log |
| Can we roll back? | Rebuild Docker with old weights | `load_model("models:/name/1")` |
| Full lineage? | Doesn't exist | Run → params + metrics + data version |

## Summary

| Concept | What it does |
|---------|-------------|
| `mlflow.register_model()` | Add a trained model to the registry |
| `transition_model_version_stage()` | Move model between stages |
| `mlflow.pytorch.load_model("models:/name/stage")` | Load model by name, not path |
| Version numbers | Auto-increment (v1, v2, v3...) |
| Descriptions | Annotate models for audits |

**Key takeaway:** A model registry replaces "which `.pth` file is the right one?" with a proper catalog that has versions, stages, and full traceability back to the training run.

**Next:** DVC Pipelines — automating the full workflow (data → train → evaluate).